## 1. Installation

In [1]:
!pip install -q --upgrade "markitdown[pdf]"
from importlib.metadata import version
print("markitdown version :", version("markitdown"))
print("pdfplumber version :", version("pdfplumber"))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 612.1 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 53.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 57.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.0/70.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 45.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 64.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 64.4 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2

## 2. Verifier le dataset Kaggle

Sur Kaggle, les fichiers d'un dataset ajoute au notebook sont montes en lecture seule sous `/kaggle/input/<nom-du-dataset>/`. On liste le contenu pour confirmer le chemin exact avant de continuer (utile si le PDF est dans un sous-dossier).

In [2]:
import os

# Etape 1 : lister tous les datasets disponibles sous /kaggle/input
print("Datasets disponibles dans /kaggle/input :")
for name in os.listdir("/kaggle/input"):
    print(" -", name)

Datasets disponibles dans /kaggle/input :
 - datasets


In [3]:
# Etape 2 : exploration recursive complete pour trouver tous les PDF,
# quelle que soit la profondeur des sous-dossiers
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.lower().endswith(".pdf"):
            print(os.path.join(root, f))

/kaggle/input/datasets/hasnaasaadi/my-dataset6/loan_0001.pdf


## 3. Configuration

**Difference fondamentale avec EasyOCR / PaddleOCR / Tesseract** : ni MarkItDown ni `pdfplumber` ne font d'OCR
sur des images. Ils lisent directement le contenu texte deja encode dans le PDF, comme Docling en mode
`do_ocr=False`. Cela n'a de sens que si le PDF est un **PDF texte natif** (non scanne) — ce qui est le cas de
nos documents synthetiques. Consequence pratique : pas de conversion en images, pas de score de confiance, pas
de bounding box au sens OCR (mais des positions de texte, utilisees pour la reconstruction de tableau).

**Tableaux** : le convertisseur PDF natif de MarkItDown (>= 0.1.5) sait detecter des tableaux **sans bordure**
via une analyse statistique des espacements de colonnes. Mais nos PDF de credit ont de vraies bordures
dessinees, un cas que cette heuristique ne couvre pas (teste et confirme : le texte ressort dans l'ordre du
flux, sans structure de tableau).

**Solution retenue** : on contourne MarkItDown pour la structure et on utilise directement `pdfplumber`
(deja installe comme dependance de MarkItDown), qui detecte les tableaux via leurs vraies bordures avec
`page.find_tables()`. Le texte hors tableau (titres, paragraphes) est extrait separement et les deux sont
reassembles dans l'ordre vertical d'origine de la page.

In [5]:
from markitdown import MarkItDown
import pdfplumber

converter = MarkItDown()
print("Converter MarkItDown pret ")
print("pdfplumber pret")

Converter MarkItDown pret 
pdfplumber pret


## 4. Lecture du PDF

Chemin adapte a Kaggle : `/kaggle/input/<dataset>/...` pour la lecture (lecture seule). Pas de conversion en
images ici : `pdfplumber` (comme MarkItDown) lit le fichier PDF directement.

In [7]:
# Adapte le nom de fichier si besoin, d'apres la sortie de la cellule 2
DATASET_DIR = "/kaggle/input/datasets/hasnaasaadi/my-dataset6"
PDF_PATH = os.path.join(DATASET_DIR, "loan_0001.pdf")

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("PDF a traiter :", PDF_PATH)

PDF a traiter : /kaggle/input/datasets/hasnaasaadi/my-dataset6/loan_0001.pdf


## 4b. Conversion MarkItDown brute (a titre de comparaison)

Sert de reference pour montrer la limite du convertisseur natif sur ce type de PDF (tableaux a bordures).

In [8]:
result = converter.convert(PDF_PATH)
md_text_brut = getattr(result, "text_content", None) or getattr(result, "markdown", None)

print(md_text_brut)

Loan Application for a Commercial Real Estate
Loan - Company Profile

1. Applicant

Company Name

Legal Form

Söding GmbH & Co. KGaA UG

Limited Partnership (KG)

Date of Incorporation

01/12/2006

Business Address

Falk-Textor-Weg 160, 86608 Gransee, Germany

Commercial Register Number / Court

HRB 977141 / Gransee Local Court

VAT ID / Tax Number

DE681242882

2. Financial Request / Property Description

Type of Property

Property Name

Adress

Office and Commercial Building

Werden Center Gransee

Laszlo-Hiller-Allee 29, 79030 Gransee, Germany

Purchase Price / Construction Costs

€1,337,000

Desired Financing Amount

€944,000

Purpose of Use

Equity Contribution

Year of Construction

Total Area

3. Financing Proposal

Desired Loan Amount

Term

Modernization

€393,000

1990

3,133 m²

€944,000

25 years

Preferred Installment Amount

€4,100 per month

Interest Rate

Early Repayment Desired?

Public Subsidies Applied For?

Variable

[ ] yes [x] no

[x] yes [ ] no

4. Declaration & 

## 5. Reconstruction avec detection de tableaux (pdfplumber)

Pour chaque page :
1. `page.find_tables()` detecte les tableaux via leurs bordures dessinees.
2. Le texte hors des zones de tableau est extrait mot par mot (`page.extract_words()`) et regroupe en lignes.
3. Blocs de texte et tableaux sont trie par position verticale (`top`) pour respecter l'ordre de lecture,
   puis les tableaux sont formates en Markdown (`| ... | ... |`).

In [10]:
def bbox_overlap(word, table_bbox):
    x0, top, x1, bottom = table_bbox
    return not (word['x1'] < x0 or word['x0'] > x1 or word['bottom'] < top or word['top'] > bottom)

md_lines = []
with pdfplumber.open(PDF_PATH) as pdf:
    for page_num, page in enumerate(pdf.pages, start=1):
        md_lines.append(f"\n\n## Page {page_num}\n")

        tables = page.find_tables()
        table_bboxes = [t.bbox for t in tables]

        # Texte hors tableaux
        words = page.extract_words()
        free_words = [w for w in words if not any(bbox_overlap(w, b) for b in table_bboxes)]

        lines_by_top = {}
        for w in free_words:
            key = round(w['top'])
            lines_by_top.setdefault(key, []).append(w)

        free_blocks = []
        for top, ws in lines_by_top.items():
            ws_sorted = sorted(ws, key=lambda w: w['x0'])
            text = " ".join(w['text'] for w in ws_sorted)
            free_blocks.append({"top": top, "type": "text", "content": text})

        table_blocks = []
        for t in tables:
            data = t.extract()
            table_blocks.append({"top": t.bbox[1], "type": "table", "content": data})

        blocks = sorted(free_blocks + table_blocks, key=lambda b: b["top"])

        for b in blocks:
            if b["type"] == "text":
                if b["content"].strip():
                    md_lines.append(b["content"])
            else:
                data = [row for row in b["content"] if row]
                if not data:
                    continue
                header = data[0]
                md_lines.append("| " + " | ".join(c or "" for c in header) + " |")
                md_lines.append("|" + "|".join(["---"] * len(header)) + "|")
                for row in data[1:]:
                    md_lines.append("| " + " | ".join((c or "").replace("\n", " ") for c in row) + " |")

md_text = "\n".join(md_lines)
print(md_text)



## Page 1

Loan Application for a Commercial Real Estate
Loan - Company Profile
1. Applicant
| Company Name | Söding GmbH & Co. KGaA UG |
|---|---|
| Legal Form | Limited Partnership (KG) |
| Date of Incorporation | 01/12/2006 |
| Business Address | Falk-Textor-Weg 160, 86608 Gransee, Germany |
| Commercial Register Number / Court | HRB 977141 / Gransee Local Court |
| VAT ID / Tax Number | DE681242882 |
2. Financial Request / Property Description
| Type of Property | Office and Commercial Building |
|---|---|
| Property Name | Werden Center Gransee |
| Adress | Laszlo-Hiller-Allee 29, 79030 Gransee, Germany |
| Purchase Price / Construction Costs | €1,337,000 |
| Desired Financing Amount | €944,000 |
| Purpose of Use | Modernization |
| Equity Contribution | €393,000 |
| Year of Construction | 1990 |
| Total Area | 3,133 m² |
3. Financing Proposal
| Desired Loan Amount | €944,000 |
|---|---|
| Term | 25 years |
| Preferred Installment Amount | €4,100 per month |
| Interest Rate | Va

## 6. Export du contenu

In [11]:
# Sauvegarde de la version brute MarkItDown (reference)
md_brut_out_path = os.path.join(OUTPUT_DIR, "loan_0001_markitdown_brut.md")
with open(md_brut_out_path, "w", encoding="utf-8") as f:
    f.write(md_text_brut)
print("Sauvegarde OK :", md_brut_out_path)

# Sauvegarde de la version avec tableaux reconstruits (pdfplumber)
md_out_path = os.path.join(OUTPUT_DIR, "loan_0001_markitdown_table.md")
with open(md_out_path, "w", encoding="utf-8") as f:
    f.write(md_text)
print("Sauvegarde OK :", md_out_path)

Sauvegarde OK : /kaggle/working/loan_0001_markitdown_brut.md
Sauvegarde OK : /kaggle/working/loan_0001_markitdown_table.md


## Resume des chemins Kaggle

- Lecture (input, lecture seule) : `/kaggle/input/<dataset>/...`
- Ecriture (output) : `/kaggle/working/...` -> c'est ce dossier qui apparait ensuite dans l'onglet "Output" du notebook Kaggle une fois execute/commit

## Resume des parametres principaux

- Pas d'OCR, pas de conversion en images : extraction directe du texte du PDF natif
- MarkItDown seul (`converter.convert`) ne reconstruit pas les tableaux a bordures de nos PDF -> conserve a titre de reference (`loan_0001_markitdown_brut.md`)
- Detection de tableaux fiable via `pdfplumber.find_tables()` (bordures reelles) -> `loan_0001_markitdown_table.md`
- Alternative cloud disponible (etape 8) si un autre type de document (tableaux sans bordure) necessite Azure Document Intelligence
- Sorties utiles : `loan_0001_markitdown_brut.md` (texte brut sans tableau), `loan_0001_markitdown_table.md` (avec tableaux Markdown), `df_tables` (pandas, un DataFrame par tableau detecte)